# Data Preparation — ISIC 2018
Run this notebook **locally** to prepare all three datasets from the raw ISIC 2018 files.
Outputs are written to `dataset/yolo`, `dataset/medsam`, and `dataset/attributes`
inside the project root — ready to be uploaded to Kaggle as a dataset.

| Step | What it does |
|---|---|
| 1 | YOLO bounding-box dataset (images + label `.txt` files + `dataset.yaml`) |
| 2 | MedSAM lesion dataset (1024×1024 `.npz` files, cropped to GT bbox) |
| 3 | Attribute segmentation dataset (`.npz` files with 5-channel attribute masks) |

In [1]:
# ── Python path: add src/ so local imports work ─────────────────────────
import sys
from pathlib import Path

SRC_DIR = Path("__file__").resolve().parent   # .../ISIC-2018/src
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print(f"src on path: {SRC_DIR}")

src on path: /home/nghia/Downloads/ISIC-2018/src


---
## 1. Verify raw data paths

In [2]:
from yolo.config   import TASK1_2_INPUT, TASK1_GT
from medsam.config import TASK2_GT

for label, path in [
    ("Task1-2 images", TASK1_2_INPUT),
    ("Task1 GT masks", TASK1_GT),
    ("Task2 GT attrs ", TASK2_GT),
]:
    count = len(list(path.glob("*"))) if path.exists() else 0
    status = "✓" if path.exists() else "✗ NOT FOUND"
    print(f"  {status}  {label}: {path}  ({count} files)")

  ✓  Task1-2 images: /home/nghia/Downloads/ISIC-2018/ISIC2018_Task1-2_Training_Input  (2596 files)
  ✗ NOT FOUND  Task1 GT masks: /home/nghia/Downloads/ISIC-2018/ISIC2018_Task1_Training_GroundTruth  (0 files)
  ✓  Task2 GT attrs : /home/nghia/Downloads/ISIC-2018/ISIC2018_Task2_Training_GroundTruth_v3  (12972 files)


---
## 2. Prepare YOLO dataset
Converts segmentation masks → YOLO bounding-box labels and writes `dataset.yaml`.

In [ ]:
from yolo.prepare_data import get_image_ids, split_ids, prepare_yolo

image_ids = get_image_ids()
print(f"Found {len(image_ids)} images with segmentation masks")

train_ids, val_ids = split_ids(image_ids)
print(f"Split: {len(train_ids)} train / {len(val_ids)} val")

prepare_yolo(train_ids, val_ids)

---
## 3. Prepare MedSAM lesion dataset
Crops each image to the GT lesion bbox (+ margin), resizes to 1024×1024, and saves as `.npz`.

In [ ]:
from medsam.prepare_data import get_image_ids, split_ids, prepare_medsam

image_ids = get_image_ids()
train_ids, val_ids = split_ids(image_ids)
print(f"Split: {len(train_ids)} train / {len(val_ids)} val")

prepare_medsam(train_ids, val_ids)

---
## 4. Prepare attribute segmentation dataset
Saves full-resolution images + 5-channel attribute masks + lesion mask as `.npz`.

In [ ]:
from medsam.prepare_data import get_image_ids, split_ids, prepare_attributes

image_ids = get_image_ids()
train_ids, val_ids = split_ids(image_ids)
print(f"Split: {len(train_ids)} train / {len(val_ids)} val")

prepare_attributes(train_ids, val_ids)

---
## 5. Verify output

In [ ]:
from yolo.config   import YOLO_DIR
from medsam.config import MEDSAM_DIR, ATTR_DIR

for label, path in [
    ("YOLO  train images", YOLO_DIR / "images" / "train"),
    ("YOLO  val   images", YOLO_DIR / "images" / "val"),
    ("YOLO  train labels", YOLO_DIR / "labels" / "train"),
    ("MedSAM lesion train", MEDSAM_DIR / "train"),
    ("MedSAM lesion val  ", MEDSAM_DIR / "val"),
    ("Attr   train       ", ATTR_DIR   / "train"),
    ("Attr   val         ", ATTR_DIR   / "val"),
]:
    count = len(list(path.glob("*"))) if path.exists() else 0
    print(f"  {label}: {count} files")